# 🛡️ JudolGuard — Training Pipeline (v2 — Bug Fixed)
## Natural Language Processing | Kelompok 13

---

**Versi ini memperbaiki 2 bug yang ditemukan pada pengujian sebelumnya:**

| Bug | Komentar | Sebelum | Sesudah |
|-----|----------|---------|---------|
| 1 | "Dilarang keras bermain judol!" | 🚫 Judol (salah) | ✅ Aman (benar) |
| 2 | "Nyesel main di situs lain, mending di situs aku" | ✅ Aman (salah) | 🚫 Judol (benar) |

**Akar masalah:**
- Bug 1: kata `dilarang/terlarang/larang` belum ada di pola ANTI_RE, sehingga jatuh ke SVM yang ternyata salah memprediksi.
- Bug 2: pola ANTI_RE terlalu general — hanya mengecek kata `nyesel` berdiri sendiri, padahal kalimat ini sebenarnya **promosi terselubung**.


## 1. Import Library

In [ ]:
import pandas as pd
import numpy as np
import re
import pickle
import os
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, classification_report
)

print('Semua library berhasil diimport')


---
## 2. Load Dataset


In [ ]:
train_orig = pd.read_csv('judol_train_dataset.csv')
test_orig  = pd.read_csv('judol_test_dataset.csv')
xlsx       = pd.read_excel('judi_online_comments.xlsx')

print(f'Train : {len(train_orig):,} baris')
print(f'Test  : {len(test_orig):,} baris')
print(f'XLSX  : {len(xlsx):,} baris')


---
## 3. Smart Relabeling — Regex v2 (Fixed)

### Perubahan dari versi sebelumnya

**PROMO_RE** — ditambah pola *covert recruitment*: `main di situs aku/ku/sini` (menyamar sebagai curhat, sebenarnya promosi).

**ANTI_RE** — ditambah kata larangan `dilarang/terlarang/larang`; kata `nyesel/menyesal/kapok` diperketat agar hanya valid jika eksplisit menyebut konteks judi (bukan berdiri sendiri).


In [ ]:
PROMO_RE = re.compile(
    r'\balex\w*17\b|4lexis\w*17|\bmandalika\s*77\b|\bweton\s*88\b|\bpulau\s*77\w\b|\bpulauwin\b|'
    r'\bsl[o0]t\b.{0,50}\b(gacor|maxwin|terpercaya|daftar|deposit|bonus|scatter|x\d+)\b|'
    r'\b(gacor|maxwin|terpercaya|daftar|deposit|bonus|scatter)\b.{0,50}\bsl[o0]t\b|'
    r'\b(daftar|gabung|join|dm|hubungi|chat|klik|link)\b.{0,40}\b(slot|judi|judol|gacor|maxwin|admin|bio|wa)\b|'
    r'\b(slot|judi|judol|gacor)\b.{0,40}\b(daftar|gabung|join|dm|hubungi|klik|link|bio|wa|admin)\b|'
    r'\bm[4a]xw[1i]n\b|\bsl[o0]t\s+g[a4]c[o0]r\b|\bg[a4]c[o0]r\s+sl[o0]t\b|'
    r'\b(deposit|depo|wd|withdraw)\b.{0,40}\b(slot|judi|gacor|maxwin|terpercaya)\b|'
    r'\b(modal\s*kecil|rezeki\s*mengalir|anti\s*rungkad|gampang\s*jepe|gampang\s*menang)\b|'
    r'\b[sS]\s+[lL]\s+[oO]\s+[tT]\b|\b[jJ]\s+[uU]\s+[dD]\s+[iI]\b|\b[gG]\s+[aA]\s+[cC]\s+[oO]\s+[rR]\b|'
    r'\b(zeus|olympus|starlight\s*princess|scatter\s*hitam|pgsoft|pragmatic)\b.{0,30}\b(x\d+|gacor|maxwin|bocoran)\b|'
    r'\bbocoran\s+(slot|gacor|hari\s*ini|malam\s*ini)\b|'
    r'\blink\s+(gacor|slot|judi|judol|terpercaya)\b|'
    r'\b(buruan|langsung|sekarang|hari\s*ini)\b.{0,30}\b(daftar|gabung|slot|gacor|maxwin)\b|'
    r'\bmain\s+di\s+(situs|tempat)\s+(aku|ku|saya|sini|gue|punyaku)\b|'
    r'\b(situs|tempat)\s+(aku|ku|saya|punyaku)\b.{0,30}\b(main|gacor|menang|terpercaya)\b',
    re.IGNORECASE
)

ANTI_RE = re.compile(
    r'\b(haram|bahaya|dosa|rugi|bangkrut|hancur|rusak|tobat|insaf|kapok)\b|'
    r'\b(dilarang|terlarang|jangan|stop|hindari|tinggalkan|berhenti|keluar|lepas|larang)\b.{0,50}\b(judi|judol|slot|gambling|main)\b|'
    r'\b(judi|judol|slot|gambling)\b.{0,50}\b(dilarang|terlarang|jangan|stop|hindari|haram|dosa|bahaya|rugi|bangkrut|larang)\b|'
    r'\b(berantas|memberantas|hapus|tutup|blokir|perangi|lawan|basmi)\b.{0,50}\b(judi|judol|slot)\b|'
    r'\b(judi|judol|slot)\b.{0,50}\b(berantas|hapus|tutup|blokir|perangi|lawan|basmi)\b|'
    r'\bkorban\s+(judi|judol|slot)\b|\b(judi|judol|slot)\s+korban\b|'
    r'\b(merusak|menghancurkan|merugikan).{0,40}\b(keluarga|rumah\s*tangga|hidup|masa\s*depan)\b|'
    r'\binget\s+mati\b|\bingat\s+akhirat\b|\bsemoga\s+(sadar|tobat|insaf|berhenti)\b|'
    r'\b(awas|waspada)\b.{0,30}\b(judi|judol|slot)\b|'
    r'\b(nyesel|menyesal|kapok)\b.{0,40}\b(judi|judol|slot|gacor|maxwin)\b|'
    r'\b(judi|judol|slot|gacor|maxwin)\b.{0,40}\b(nyesel|menyesal|kapok)\b',
    re.IGNORECASE
)

def smart_label(text, orig_label):
    txt      = str(text)
    is_promo = bool(PROMO_RE.search(txt))
    is_anti  = bool(ANTI_RE.search(txt))
    if is_anti and not is_promo:  return 0
    if is_promo and not is_anti:  return 1
    if is_anti and is_promo:      return 0
    return orig_label

bug_cases = [
    ('Dilarang keras bermain judol!', 0),
    ('Nyesel deh main di situs lain, mending main di situs aku', 1),
]
print('Verifikasi fix bug:')
for txt, exp in bug_cases:
    got = smart_label(txt, -1)
    mark = 'OK' if got==exp else 'GAGAL'
    print(f'  [{mark}] exp={exp} got={got} | "{txt}"')


---
## 4. Preprocessing Teks

Ditambahkan tahap penghapusan stopword Bahasa Indonesia setelah normalisasi slang.
Tahap ini perlu karena beberapa kata fungsi seperti `lagi`, `yang`, `kalo`, `berarti`
ternyata muncul jauh lebih sering pada komentar berlabel judol di data latih
(gaya tulisan komentar yang lebih panjang/naratif), sehingga TF-IDF salah
menganggapnya sebagai sinyal topik padahal kata tersebut netral.


In [ ]:
NORM = {
    'yg':'yang', 'dgn':'dengan', 'gk':'tidak', 'ga':'tidak', 'gak':'tidak',
    'nggak':'tidak', 'ngga':'tidak', 'krn':'karena', 'karna':'karena',
    'utk':'untuk', 'tdk':'tidak', 'tp':'tapi', 'lg':'lagi', 'udh':'sudah',
    'sdh':'sudah', 'dg':'dengan', 'sm':'sama', 'jd':'jadi', 'aja':'saja',
    'bgt':'banget', 'emg':'memang', 'emang':'memang', 'bs':'bisa', 'org':'orang',
    'dr':'dari', 'klo':'kalau', 'kl':'kalau', 'sy':'saya', 'gw':'saya',
    'gue':'saya', 'lo':'kamu', 'lu':'kamu', 'wkwkwk':'tertawa', 'wkwk':'tertawa',
    'sl0t':'slot', 's1ot':'slot', 'jud1':'judi', 'b0coran':'bocoran', 'maxw1n':'maxwin',
    'g4cor':'gacor', 'g4c0r':'gacor', 'gac0r':'gacor', '4lexis':'alexis',
    'alex1s':'alexis', 'm4xwin':'maxwin',
}

STOPWORDS_ID = set("""
yang dan di ke dari pada untuk dengan atau ini itu juga lagi saja akan
sudah belum masih bisa ada tidak tak ga gak engga nggak ngga jadi karena
karna kalo kalau klo kl jika kl bila maka tapi tp namun walau walaupun
biar agar supaya hanya cuma cuman pun lah kah deh dong sih nih situ sini
sana mereka kita kami saya aku gw gue lo lu kamu anda dia ia nya mu ku
pak bu bang kak adek mas mbak om tante banget bgt sekali amat terlalu
paling lebih kurang antara sambil sampai hingga sebelum sesudah setelah
ketika saat waktu tahun bulan hari jam menit detik nya an kan in
""".split())

def normalize_spaced(text):
    def collapse(m): return re.sub(r'[\s._-]', '', m.group(0))
    return re.sub(r'(?<!\w)([a-zA-Z](?:[\s._-][a-zA-Z]){2,})(?!\w)', collapse, text)

def preprocess(text):
    if not text or str(text).strip() == '': return ''
    text = str(text).encode('ascii', 'ignore').decode('ascii')
    text = text.lower()
    text = normalize_spaced(text)
    for s,d in [('0','o'),('1','i'),('3','e'),('4','a'),('5','s')]:
        text = text.replace(s, d)
    text = re.sub(r'http\S+|www\S+', '', text)
    text = re.sub(r'@\w+|#\w+', '', text)
    text = re.sub(r'\d+', '', text)
    text = re.sub(r'[^a-z\s]', ' ', text)
    words = [NORM.get(w, w) for w in text.split()]
    words = [w for w in words if w not in STOPWORDS_ID]
    return re.sub(r'\s+', ' ', ' '.join(words)).strip()

print(preprocess('Dilarang keras bermain judol!'))
print(preprocess('s l o t gacor maxwin terpercaya'))
print(preprocess('lagi nyeritain'))
print(preprocess('berarti kamu smp 3'))


---
## 5. Bangun Dataset Final + TF-IDF Feature Extraction


In [ ]:
train = train_orig.copy()
train['label'] = train_orig.apply(lambda r: smart_label(r['text'], r['label']), axis=1)
test = test_orig.copy()
test['label'] = test_orig.apply(lambda r: smart_label(r['text'], r['label']), axis=1)

JUDOL_XLSX = re.compile(
    r'alexis.*17|mandalika.*77|weton.*88|pulau.*777|pulauwin|'
    r'sl[o0]t|g[a4]c[o0]r|max\s*win|scatter|zeus|olympus|pgsoft|pragmatic|'
    r'jepe|\bjp\b|depo(sit)?|\bwd\b|withdraw|bonus|terpercaya|bocoran|'
    r'rungkad|gampang|landing|cair|gabung|daftar|situs|\b(77|88|777|17)\b',
    re.IGNORECASE
)
xlsx_rows = []
for _, row in xlsx.iterrows():
    txt = str(row['komentar'])
    lbl = 1 if JUDOL_XLSX.search(txt) else 0
    lbl = smart_label(txt, lbl)
    xlsx_rows.append({'text': txt, 'label': lbl})
df_xlsx = pd.DataFrame(xlsx_rows)

aug_pairs = [
    ('s l o t gacor hari ini', 1), ('j u d i online terpercaya', 1),
    ('bocoran slot malam ini link di bio', 1), ('slot gacor maxwin daftar sekarang', 1),
    ('alexis17 gampang menang terpercaya', 1), ('weton88 scatter gacor hari ini', 1),
    ('zeus x500 gampang jepe maxwin', 1), ('modal kecil untung besar slot', 1),
    ('anti rungkad bocoran slot gacor', 1), ('gabung sekarang slot terpercaya', 1),
    ('dm admin slot gacor hari ini', 1), ('link slot gacor di bio', 1),
    ('deposit murah wd cepat slot online', 1), ('gacor maxwin terpercaya daftar link bio', 1),
    ('s.l.o.t gacor maxwin', 1), ('scatter hitam zeus pragmatic gacor', 1),
    ('nyesel main di situs lain mending main di situs aku', 1),
    ('main di situs aku aja lebih gacor dan terpercaya', 1),
    ('rugi main di tempat lain coba situs ku', 1),
    ('jangan main di situs sebelah, mending ke situs aku', 1),
    ('kapok main ditempat lain yuk gabung situs aku', 1),
    ('situs aku lebih gacor daripada yang lain', 1),
    ('mending main di situs ku aja dijamin menang', 1),
    ('buruan daftar sebelum ditutup link di bio', 1),
    ('jangan sampai ketinggalan daftar sekarang gacor', 1),
    ('slotking69 gas join sekarang', 1),
    ('Jangan judol, judol haram', 0), ('judi memang bahaya buat keluarga', 0),
    ('Ingat pak judi itu haram', 0), ('berantas judi online sekarang', 0),
    ('saya korban slot tutup situs slot', 0), ('slot haram slot haram', 0),
    ('hapus judi online dari indonesia', 0), ('judi online merusak keluarga', 0),
    ('stop judol sekarang juga', 0), ('tobat dari judi sebelum terlambat', 0),
    ('judi itu dosa besar ingat keluarga', 0), ('awas bahaya judi online', 0),
    ('pemerintah harus berantas judi online', 0), ('ayo hapus judi online di indonesia', 0),
    ('jangan main judi itu haram dan bahaya', 0), ('berhenti main slot itu rugi', 0),
    ('judi menghancurkan rumah tangga', 0), ('korban judi online makin banyak', 0),
    ('semoga sadar berhenti dari judi', 0), ('ingat mati jangan judi', 0),
    ('Dilarang keras bermain judol', 0), ('dilarang bermain slot di sini', 0),
    ('terlarang ikut judi online', 0), ('dilarang keras ikut judi online apapun', 0),
    ('jangan sekali kali coba judi online', 0), ('dilarang oleh agama bermain judi', 0),
    ('larangan keras terhadap judi online', 0), ('dilarang pemerintah bermain judol', 0),
    ('nyesel banget kalah terus main slot', 0), ('menyesal sudah coba judi online', 0),
    ('kapok main slot rugi banyak', 0), ('nyesel ikut judi malah bangkrut', 0),
    ('terima kasih tutorialnya sangat membantu', 0), ('videonya bagus banget kak', 0),
    ('semangat terus upload kontennya', 0), ('mantap bang lanjutkan', 0),
    ('keren banget kontennya informatif', 0), ('musik pembukanya keren', 0),
    ('sukses selalu ya kak', 0), ('penjelasannya mudah dipahami', 0),
    ('lagi nyeritain pengalaman', 0), ('lagi santai aja di rumah', 0),
    ('berarti kamu masih sekolah ya', 0), ('berarti dia salah paham', 0),
    ('buseeett rame banget komentarnya', 0), ('yang ganggu itu spammer bukan aku', 0),
    ('maap kalo salah ngomong', 0), ('maap kalo komen ku kurang jelas', 0),
    ('matiin komen aja kalo ga suka', 0), ('matiin notif biar gak ganggu', 0),
    ('knp kamu gitu sih', 0), ('knp jadi rame begini', 0),
    ('pilih mana susah juga ya', 0), ('mau tanya aja sih sebenarnya', 0),
    ('contoh aja sih kalo gitu', 0), ('komen di tiktok emang rame banget', 0),
    ('aduh mau tanya soal hukum aja', 0), ('suami aku tiap hari kerja', 0),
    ('diam aja deh daripada ribut', 0), ('gas kita jalan jalan aja', 0),
]
df_aug = pd.DataFrame(aug_pairs, columns=['text','label'])

train_all = pd.concat([train[['text','label']], df_xlsx, df_aug], ignore_index=True)
train_all['clean'] = train_all['text'].apply(preprocess)
test['clean']      = test['text'].apply(preprocess)
train_all = train_all[train_all['clean'].str.len() > 0].reset_index(drop=True)
test      = test[test['clean'].str.len() > 0].reset_index(drop=True)

print(f'Train final : {len(train_all):,} baris | {train_all["label"].value_counts().to_dict()}')
print(f'Test        : {len(test):,} baris | {test["label"].value_counts().to_dict()}')

tfidf = TfidfVectorizer(max_features=12000, ngram_range=(1,2), min_df=2, sublinear_tf=True)
X_train = tfidf.fit_transform(train_all['clean'])
X_test  = tfidf.transform(test['clean'])
y_train = train_all['label']
y_test  = test['label']

print(f'\nTF-IDF vocabulary: {len(tfidf.vocabulary_):,} fitur')
print(f'Shape Train matrix: {X_train.shape}')
print(f'Shape Test  matrix: {X_test.shape}')


---
## 6. Training 4 Model ML


In [ ]:
models_dict = {
    'Logistic Regression': LogisticRegression(C=1.0, max_iter=1000, random_state=42),
    'SVM'                : LinearSVC(C=1.0, max_iter=2000, random_state=42),
    'Naive Bayes'        : MultinomialNB(alpha=0.1),
    'Random Forest'      : RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
}

eval_results   = {}
trained_models = {}

for name, model in models_dict.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    eval_results[name] = {
        'accuracy' : accuracy_score(y_test, y_pred),
        'precision': precision_score(y_test, y_pred, average='weighted'),
        'recall'   : recall_score(y_test, y_pred, average='weighted'),
        'f1'       : f1_score(y_test, y_pred, average='weighted'),
        'cm'       : confusion_matrix(y_test, y_pred),
        'y_pred'   : y_pred,
    }
    trained_models[name] = model
    r = eval_results[name]
    print(f'{name:25}  Acc={r["accuracy"]*100:.2f}%  F1={r["f1"]*100:.2f}%')


---
## 7. Evaluasi & Perbandingan Model


In [ ]:
MODEL_NAMES = ['Logistic Regression','SVM','Naive Bayes','Random Forest']
summary = pd.DataFrame([{
    'Model'    : name + (' (terbaik)' if name=='SVM' else ''),
    'Accuracy' : f"{eval_results[name]['accuracy']*100:.2f}%",
    'Precision': f"{eval_results[name]['precision']*100:.2f}%",
    'Recall'   : f"{eval_results[name]['recall']*100:.2f}%",
    'F1-Score' : f"{eval_results[name]['f1']*100:.2f}%",
} for name in MODEL_NAMES])

best = max(eval_results, key=lambda n: eval_results[n]['f1'])
print(f'Model Terbaik: {best} -- F1={eval_results[best]["f1"]*100:.2f}%\n')
display(summary)


In [ ]:
metrics = ['accuracy','precision','recall','f1']
mlbls   = ['Accuracy','Precision','Recall','F1-Score']
COLORS  = ['#38bdf8','#f472b6','#34d399','#fb923c']
x, w    = np.arange(4), 0.18

fig, ax = plt.subplots(figsize=(12, 5))
for i,(name,color) in enumerate(zip(MODEL_NAMES,COLORS)):
    vals = [eval_results[name][m] for m in metrics]
    bars = ax.bar(x+i*w, vals, w, label=name, color=color, alpha=0.9)
    for bar,val in zip(bars,vals):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.004,
                f'{val*100:.1f}', ha='center', fontsize=8, fontweight='bold')

ax.set_xticks(x+w*1.5); ax.set_xticklabels(mlbls, fontsize=11)
ax.set_ylim(0.75, 1.05)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v,_: f'{v:.0%}'))
ax.legend(fontsize=10); ax.set_title('Perbandingan Performa 4 Model ML (Data v2)', fontsize=13)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout(); plt.show()


In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
for ax, name, color in zip(axes, MODEL_NAMES, COLORS):
    cm = eval_results[name]['cm']
    sns.heatmap(cm, annot=True, fmt='d', ax=ax, cmap='Blues',
                xticklabels=['Non-Judol','Judol'], yticklabels=['Non-Judol','Judol'],
                annot_kws={'size':13,'weight':'bold'})
    star = ' (terbaik)' if name=='SVM' else ''
    r = eval_results[name]
    ax.set_title(f'{name}{star}\nAcc {r["accuracy"]*100:.1f}%  F1 {r["f1"]*100:.1f}%', fontsize=10)
    ax.set_xlabel('Prediksi'); ax.set_ylabel('Aktual')
plt.tight_layout(); plt.show()


In [ ]:
print(f'Classification Report: {best}')
print(classification_report(y_test, eval_results[best]['y_pred'],
                             target_names=['Non-Judol (0)', 'Judol (1)']))


---
## 8. Simpan Model Terbaik


In [ ]:
os.makedirs('model', exist_ok=True)

save_results = {
    k: {
        'accuracy' : round(v['accuracy'], 4),
        'precision': round(v['precision'], 4),
        'recall'   : round(v['recall'], 4),
        'f1'       : round(v['f1'], 4),
        'cm'       : v['cm'].tolist(),
    }
    for k, v in eval_results.items()
}

pickle.dump({
    'tfidf'        : tfidf,
    'models'       : trained_models,
    'eval_results' : save_results,
    'best_model'   : best,
}, open('model/judol_model.pkl', 'wb'))

print(f'Model disimpan ke: model/judol_model.pkl')
print(f'Best model : {best}')
print(f'F1-Score   : {eval_results[best]["f1"]*100:.2f}%')
print(f'Accuracy   : {eval_results[best]["accuracy"]*100:.2f}%')


---
## 9. Demo Prediksi Hybrid (Regex + SVM)


In [ ]:
KRITIK_HUKUM_RE = re.compile(
    r'\btangkap\b.{0,40}\b(bandar|artis|penjudi|streamer|pelaku|admin)\b|'
    r'\b(bandar|artis|penjudi|streamer|pelaku)\b.{0,40}\btangkap\b|'
    r'\bdiberantas\b.{0,40}\b(situs|bandar|judi)\b|'
    r'\b(situs|bandar|judi)\b.{0,40}\bdiberantas\b|'
    r'\bdi\s*brantas\b.{0,40}\b(judi|situs|bandar)\b|'
    r'\b(judi|situs|bandar)\b.{0,40}\bdi\s*brantas\b|'
    r'\busut\w*\b.{0,80}\b(judi|kasus|bandar)\b|'
    r'\b(judi|kasus|bandar)\b.{0,80}\busut\w*\b|'
    r'\bditangkap\b.{0,60}\bpromosi\b.{0,40}\b(judi|judol|slot)\b|'
    r'\bpromosi\b.{0,40}\b(judi|judol|slot)\b.{0,60}\bditangkap\b',
    re.IGNORECASE | re.DOTALL
)

CURHAT_RE = re.compile(
    r'\bketemu\s+penjudi\b|\bkenal\s+penjudi\b|\bsuami\s+(tukang\s+)?judi\b|'
    r'\bistri\s+(tukang\s+)?judi\b|\bnasehat\w*\b.{0,80}\bpenjudi\b|'
    r'\bpenjudi\b.{0,80}\bnasehat\w*\b|\bsusah\s+dinasehat\w*\b|'
    r'\bbreak\s+dulu\b.{0,30}\bpenjudi\b|\bmusuhi\b.{0,40}\bpenjudi\b|'
    r'\bbebal\b.{0,40}\bpenjudi\b|\bpenjudi\b.{0,40}\bbebal\b',
    re.IGNORECASE | re.DOTALL
)

DISKUSI_BERITA_RE = re.compile(
    r'\bbansos\b.{0,100}\b(judi|judol|penjudi)\b|\b(judi|judol|penjudi)\b.{0,100}\bbansos\b|'
    r'\bkasus\b.{0,80}\b(judi|judol)\b|\b(judi|judol)\b.{0,80}\bkasus\b|'
    r'\b\d+\s*[tT]\b.{0,40}\bjudi\b|\bjudi\b.{0,40}\b\d+\s*[tT]\b|'
    r'\bpajak\b.{0,30}\bjudi\b|\bjudi\b.{0,30}\bpajak\b|'
    r'\bkorupsi\b.{0,40}\bjudi\b|\bjudi\b.{0,40}\bkorupsi\b',
    re.IGNORECASE | re.DOTALL
)

INTENT_LABELS = {
    'kritik_hukum': 'Kritik/Penegakan Hukum',
    'curhat': 'Curhat/Pengalaman Pribadi',
    'diskusi_berita': 'Diskusi Berita/Isu Sosial',
}
INTENT_CONF = 0.90
SVM_DOUBT_THRESHOLD = 0.70

def predict_hybrid(text, model, vectorizer):
    orig     = str(text)
    clean    = preprocess(orig)
    is_anti  = bool(ANTI_RE.search(orig))
    is_promo = bool(PROMO_RE.search(orig))

    is_kh = bool(KRITIK_HUKUM_RE.search(orig))
    is_ch = bool(CURHAT_RE.search(orig))
    is_db = bool(DISKUSI_BERITA_RE.search(orig))
    intent_key = 'kritik_hukum' if is_kh else ('curhat' if is_ch else ('diskusi_berita' if is_db else None))
    intent = INTENT_LABELS.get(intent_key) if intent_key else None

    if is_anti and not is_promo:
        return 0, 0.97, 'regex-anti', None

    if is_promo and not is_anti:
        base_label, base_conf, base_src = 1, 0.97, 'regex-promo'
    else:
        vec   = vectorizer.transform([clean])
        base_label = int(model.predict(vec)[0])
        if hasattr(model, 'decision_function'):
            raw  = model.decision_function(vec)[0]
            base_conf = float(1 / (1 + np.exp(-abs(raw))))
        else:
            base_conf = 1.0
        base_src = 'svm'
        if is_anti and is_promo:
            base_label = 0

    if intent is not None:
        came_from_promo_regex = (base_src == 'regex-promo')
        svm_is_doubtful = base_conf < SVM_DOUBT_THRESHOLD
        intent_stronger_than_svm = (base_src == 'svm') and (INTENT_CONF >= base_conf)
        if came_from_promo_regex or svm_is_doubtful or intent_stronger_than_svm:
            return 0, INTENT_CONF, f'intent-{intent_key}', intent
        return base_label, base_conf, base_src, intent

    if base_src == 'svm' and base_conf < SVM_DOUBT_THRESHOLD:
        return base_label, base_conf, base_src, 'Uncategorized'

    return base_label, base_conf, base_src, None

demo_cases = [
    ('Dilarang keras bermain judol!',                                 0),
    ('Nyesel deh main di situs lain, mending main di situs aku',      1),
    ('Jangan judol, judol haram',                                     0),
    ('judi memang bahaya',                                            0),
    ('pemerintah harus tegas memberantas judi online',                0),
    ('slot haram slot haram',                                         0),
    ('tobat dari judi sebelum terlambat',                             0),
    ('nyesel banget kalah terus main slot',                           0),
    ('daftar slot gacor maxwin terpercaya',                           1),
    ('link gacor terpercaya bosku dm admin',                          1),
    ('bocoran slot malam ini',                                        1),
    ('m4xw1n anti rungkad scatter zeus',                              1),
    ('s l o t gacor hari ini',                                       1),
    ('videonya bagus banget kak',                                     0),
    ('terima kasih tutorialnya sangat membantu',                      0),
    ('lagi nyeritain',                                                 0),
    ('berarti kamu smp 3',                                             0),
    ('buseeett yang spammm ganggu pooll',                              0),
    ('knp slot mulu',                                                  0),
    ('maap kalo salah',                                                0),
    ('matiin komen aja pak banyak judi',                               0),
    ('tangkap artis yg judi onlyn.',                                   0),
    ('Tangkap bandar nya pak ,jd ngak bakalan ada judi online yg sudah banyak menjerumuskan anak muda dan banyak terlilit hutang pinjol', 0),
    ('astaghfirullah, apa mksd mrk menyetujui para penjudi online sbg penerima bansos,apa hrs ikut judol dulu baru msk daftar bansos,makin marak sj judolnya', 0),
    ('negara lagi nutupin kasus siapa lagi si ko baru skrg di usutnya kan dh dari lama donate judi itu', 0),
    ('ada 3 tipe orang yg klo d nasehatin susah.\n1.penjudi\n2.penjudi\n3.pendukung anis', 0),
    ('duit nya ya gosah di balikin, orang nya aja yg ditangkap karena promosi judi online di media elektronik', 0),
    ('Ini baru pajak buat judi',                                       0),
    ('ko yg kasus streamer judi online gaada kabar?',                  0),
    ('sudah beberapa kali ketemu penjudi, di nasehati malah di musuhi, manusia paling bebal kalau sudah urusan sama penjudi, sama penjudi break dulu', 0),
    ('yg diberantas ya situs judinya lah, tegasnya kok ke yg promosinya. selama bandar dan situsnya masih bebas, ga akan habis itu perjudian di indonesia', 0),
    ('judi aja di brantas eh korupsi malah di dukung',                 0),
    ('gercep bnget judi online, 271 T kmna',                           0),
]

svm_model = trained_models['SVM']
print(f'{"Komentar":<55} {"Exp":>4} {"Pred":>5} {"Conf":>6}  Sumber               Intent')
print('-'*125)
correct = 0
for txt, exp in demo_cases:
    pred, conf, src, intent = predict_hybrid(txt, svm_model, tfidf)
    ok = pred == exp; correct += ok
    mark = 'OK' if ok else 'X '
    print(f'{mark} {txt[:53]:<54} {exp:>4} {pred:>5} {conf:>5.0%}  {src:<20} {intent}')

print(f'\nSkor: {correct}/{len(demo_cases)} ({correct/len(demo_cases)*100:.0f}%)')
